# 02 — CDC Diabetes Health Indicators (UCI dataset 891)

This notebook fetches the CDC survey dataset with the same UCI repository ID used by the source project. Internet access is needed for the initial fetch; the loader caches a local CSV.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
from src.preprocessing import quality_report, descriptive_statistics, candidate_clean
from src.visualization import plot_target, plot_distributions, plot_boxplots, plot_correlations, plot_missingness, plot_target_association
from src.data_loading import load_cdc, find_target
df = load_cdc(cache=True)
target = find_target(df)
print("Shape:", df.shape, "| target:", target, "| duplicate rows:", df.duplicated().sum())
display(df.head())

## Target balance and quality

Verify the target definition from the UCI dataset card. The binary CDC target combines prediabetes and diabetes in its positive class, so it is not equivalent to every other diabetes dataset.

In [ ]:
display(df[target].value_counts(dropna=False).rename("count").to_frame())
display((df[target].value_counts(normalize=True, dropna=False)*100).round(2).rename("percent").to_frame())
display(quality_report(df))
display(descriptive_statistics(df))

## Plots

Many CDC fields are integer-coded indicators, not continuous measurements. The numeric heatmap is exploratory; consider categorical association measures for deeper analysis.

In [ ]:
out = ROOT / "outputs" / "figures" / "cdc"
plot_target(df, target, out)
plot_distributions(df, target, out, limit=24)
plot_boxplots(df, target, out, limit=18)
plot_correlations(df, target, out)
plot_missingness(df, out)
plot_target_association(df, target, out)
plt.show()

## Candidate cleaning and reports

Actual NaNs are imputed for a comparison only. Clean-looking values do not prove the dataset is error-free; consult source documentation and validate ranges.

In [ ]:
processed = candidate_clean(df)
print("Missing before:", int(df.isna().sum().sum()), "after:", int(processed.isna().sum().sum()))
report_dir = ROOT / "outputs" / "reports" / "cdc"
report_dir.mkdir(parents=True, exist_ok=True)
quality_report(df).to_csv(report_dir / "data_quality.csv", index=False)
descriptive_statistics(df).to_csv(report_dir / "descriptive_statistics.csv", index=False)